# Task 62. Text Analysis

In [1]:
# ===== Setup: download the NLTK data used in this notebook =====
import nltk

for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
                 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
                 'maxent_ne_chunker', 'maxent_ne_chunker_tab', 'words', 'names', 'movie_reviews',
                 'tagsets', 'tagsets_json']:
    nltk.download(resource, quiet=True)

In [2]:
# ===== Sample text =====
text = """Natural language processing is a field of artificial intelligence that helps computers
understand human language. Pratik and Roberto wrote a tutorial in July 2020 at Towards AI in
San Francisco. They explained how computers read text, split it into words, and find the
meaning of sentences. Machine learning models learn patterns from large amounts of text data.
Today, NLP powers search engines, chatbots, translation tools and voice assistants like Siri."""
print(text)

Natural language processing is a field of artificial intelligence that helps computers
understand human language. Pratik and Roberto wrote a tutorial in July 2020 at Towards AI in
San Francisco. They explained how computers read text, split it into words, and find the
meaning of sentences. Machine learning models learn patterns from large amounts of text data.
Today, NLP powers search engines, chatbots, translation tools and voice assistants like Siri.


In [3]:
# ===== Chunking =====
# group words into noun phrases (NP) using a regular-expression grammar on the PoS tags
from nltk import pos_tag, word_tokenize, RegexpParser

sentence = "The little yellow dog barked at the big grey cat in the old garden"
tagged = pos_tag(word_tokenize(sentence))
print(tagged)

grammar = "NP: {<DT>?<JJ>*<NN>}"
chunk_parser = RegexpParser(grammar)
chunked = chunk_parser.parse(tagged)
print(chunked)

[('The', 'DT'), ('little', 'JJ'), ('yellow', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('big', 'JJ'), ('grey', 'NN'), ('cat', 'NN'), ('in', 'IN'), ('the', 'DT'), ('old', 'JJ'), ('garden', 'NN')]
(S
  (NP The/DT little/JJ yellow/JJ dog/NN)
  barked/VBD
  at/IN
  (NP the/DT big/JJ grey/NN)
  (NP cat/NN)
  in/IN
  (NP the/DT old/JJ garden/NN))


In [4]:
for subtree in chunked.subtrees(filter=lambda t: t.label() == 'NP'):
    print(' '.join(word for word, tag in subtree.leaves()))

The little yellow dog
the big grey
cat
the old garden


In [5]:
# ===== Chinking =====
# start with everything in one chunk, then remove (chink) verbs and prepositions
grammar = r"""
Chunk: {<.*>+}
       }<VBD|IN>+{
"""
chink_parser = RegexpParser(grammar)
print(chink_parser.parse(tagged))

(S
  (Chunk The/DT little/JJ yellow/JJ dog/NN)
  barked/VBD
  at/IN
  (Chunk the/DT big/JJ grey/NN cat/NN)
  in/IN
  (Chunk the/DT old/JJ garden/NN))


In [6]:
# ===== Named Entity Recognition (NER) =====
from nltk import ne_chunk

sentence = "Pratik and Roberto wrote a tutorial at Towards AI in San Francisco, California."
ner_tree = ne_chunk(pos_tag(word_tokenize(sentence)))
print(ner_tree)

(S
  (GPE Pratik/NNP)
  and/CC
  (PERSON Roberto/NNP)
  wrote/VBD
  a/DT
  tutorial/NN
  at/IN
  (ORGANIZATION Towards/NNP)
  AI/NNP
  in/IN
  (GPE San/NNP Francisco/NNP)
  ,/,
  (GPE California/NNP)
  ./.)


In [7]:
for subtree in ner_tree:
    if hasattr(subtree, 'label'):
        print(subtree.label(), '->', ' '.join(word for word, tag in subtree.leaves()))

GPE -> Pratik
PERSON -> Roberto
ORGANIZATION -> Towards
GPE -> San Francisco
GPE -> California


In [8]:
# NER on the whole sample text
ner_text = ne_chunk(pos_tag(word_tokenize(text)))
[(t.label(), ' '.join(w for w, _ in t.leaves())) for t in ner_text if hasattr(t, 'label')]

[('PERSON', 'Pratik'),
 ('PERSON', 'Roberto'),
 ('ORGANIZATION', 'Towards'),
 ('GPE', 'San Francisco'),
 ('PERSON', 'Machine'),
 ('ORGANIZATION', 'NLP'),
 ('PERSON', 'Siri')]

In [9]:
# ===== WordNet =====
from nltk.corpus import wordnet

synsets = wordnet.synsets('good')
for s in synsets[:5]:
    print(s.name(), '-', s.definition())

good.n.01 - benefit
good.n.02 - moral excellence or admirableness
good.n.03 - that which is pleasing or valuable or useful
commodity.n.01 - articles of commerce
good.a.01 - having desirable or positive qualities especially those suitable for a thing specified


In [10]:
print(wordnet.synset('computer.n.01').examples())
print(wordnet.synset('computer.n.01').hypernyms())

[]
[Synset('machine.n.01')]


In [11]:
# synonyms and antonyms
synonyms, antonyms = set(), set()
for syn in wordnet.synsets('good'):
    for lemma in syn.lemmas():
        synonyms.add(lemma.name())
        if lemma.antonyms():
            antonyms.add(lemma.antonyms()[0].name())

print("Synonyms:", sorted(synonyms))
print("Antonyms:", sorted(antonyms))

Synonyms: ['adept', 'beneficial', 'commodity', 'dear', 'dependable', 'effective', 'estimable', 'expert', 'full', 'good', 'goodness', 'honest', 'honorable', 'in_effect', 'in_force', 'just', 'near', 'practiced', 'proficient', 'respectable', 'right', 'ripe', 'safe', 'salutary', 'secure', 'serious', 'skilful', 'skillful', 'sound', 'soundly', 'thoroughly', 'trade_good', 'undecomposed', 'unspoiled', 'unspoilt', 'upright', 'well']
Antonyms: ['bad', 'badness', 'evil', 'evilness', 'ill']


In [12]:
# similarity between words
dog, cat, car = wordnet.synset('dog.n.01'), wordnet.synset('cat.n.01'), wordnet.synset('car.n.01')
print("dog vs cat:", dog.wup_similarity(cat))
print("dog vs car:", dog.wup_similarity(car))

dog vs cat: 0.8571428571428571
dog vs car: 0.4


In [13]:
# ===== Bag of Words =====
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

sentences = ["Jim and Pam traveled by bus.",
             "The train was late.",
             "The flight was full. Traveling by flight is expensive."]

cv = CountVectorizer()
bow = cv.fit_transform(sentences)
pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out())

,and,bus,by,expensive,flight,full,is,jim,late,pam,the,train,traveled,traveling,was
0,1,1,1,0,0,0,0,1,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0,0,1,0,1,1,0,0,1
2,0,0,1,1,2,1,1,0,0,0,1,0,0,1,1


In [14]:
# ===== TF-IDF =====
from sklearn.feature_extraction.text import TfidfVectorizer

documents = ["This is the first document.",
             "This document is the second document."]

tfidf = TfidfVectorizer()
weights = tfidf.fit_transform(documents)
pd.DataFrame(weights.toarray(), columns=tfidf.get_feature_names_out()).round(3)

,document,first,is,second,the,this
0,0.409,0.575,0.409,0.000,0.409,0.409
1,0.668,0.000,0.334,0.469,0.334,0.334


In [15]:
# ===== Document similarity with TF-IDF =====
from sklearn.metrics.pairwise import cosine_similarity

tfidf_s = TfidfVectorizer().fit_transform(sentences)
pd.DataFrame(cosine_similarity(tfidf_s), index=['s1', 's2', 's3'], columns=['s1', 's2', 's3']).round(3)

,s1,s2,s3
s1,1.000,0.000,0.078
s2,0.000,1.000,0.209
s3,0.078,0.209,1.000
